# SSTW B5 Colab GPU 冷启动运行入口

该 Notebook 只作为 Colab 入口使用。正式 records、tables、reports、artifacts 与 package manifest 由仓库脚本和 `experiments/` 模块生成。

默认 Google Drive 落盘根目录:

```text
/content/drive/MyDrive/SSTW
```

目录约定:

```text
/content/drive/MyDrive/SSTW/datasets/generative_video_prompt_suite
/content/drive/MyDrive/SSTW/runs/generative_video_model_probe_colab
/content/drive/MyDrive/SSTW/packages/generative_video_model_probe
/content/drive/MyDrive/SSTW/logs/generative_video_model_probe
```

推荐主模型: `Lightricks/LTX-Video`。Notebook 通过仓库模块调用 Diffusers `LTXPipeline`。正式语义一致性 metric 默认使用 `openai/clip-vit-base-patch32`, 通过 Transformers `CLIPModel` / `CLIPProcessor` 调用。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, json
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SSTW'
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 2. 获取仓库
# 如果你已经把仓库上传到 Colab, 将 REPO_DIR 指向上传后的目录。
# 如果你有 Git 远程地址, 填写 REPO_URL 后执行此单元。
REPO_URL = 'https://github.com/RICHAAARC/SSTW.git'  # 例如: 'https://github.com/<user>/<repo>.git'
REPO_DIR = '/content/SSTW'

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"


In [ ]:
# 3. 安装 Colab GPU 运行依赖
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf


In [ ]:
# 4. 可选 Hugging Face 认证
# 公开模型通常可以匿名下载, 但 Colab 中建议提供 HF_TOKEN 以避免限流或访问 gated 模型失败。
# 注意: token 只写入当前 Colab 环境变量, 不写入 records、tables, reports 或 package manifest。
import os
# The user wants to strictly use the environment's HF_TOKEN, so removing the interactive getpass.
# from getpass import getpass # This line is no longer needed.

try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元') from exc

# Check if HF_TOKEN is set in the environment
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.') # Clarified source
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 初始化 Drive 目录布局
from paper_workflow.notebook_utils import generative_video_model_probe_workflow as probe_workflow

layout = probe_workflow.ensure_drive_layout(DRIVE_PROJECT_ROOT)
print(json.dumps(layout, ensure_ascii=False, indent=2))


In [ ]:
# 6. 构造 prompt / seed / motion 数据集
# 该步骤只构造测试输入数据, 不执行模型测试。输出落盘到 Google Drive datasets 子目录。
cmd = probe_workflow.build_prompt_suite_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 7. 配置 B5 Colab 运行目标
# smoke: 最小冷启动验证。recommended: 更多 prompt / seed, 并打开推荐目标设置。extended: 更长运行。
PROFILE = 'pilot'
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
# cross model 必须兼容当前 colab_runtime 的 LTXPipeline 调用。若不确定, 保持为空, 记录为 not_configured。
CROSS_MODEL_ID = ''
# 正式语义 metric 使用 CLIP 文本-视频相似度。该模型为公开模型, 通常不强制 HF_TOKEN, 但提供 token 可降低限流风险。
SEMANTIC_MODEL_ID = 'openai/clip-vit-base-patch32'
SEMANTIC_FRAME_LIMIT = 8
DISABLE_SEMANTIC_METRIC = False
INCLUDE_VIDEOS_IN_PACKAGE = True


In [ ]:
# 8. 执行真实 GPU 生成测试
# 正式 records / tables / artifacts 会由 experiments 模块写入 Drive run 子目录。
cmd = probe_workflow.build_colab_runtime_command(layout, PROFILE, MODEL_ID, CROSS_MODEL_ID)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 9. 执行正式质量、运动与语义度量
# 该步骤读取已生成 mp4 文件和 prompt suite, 写出 formal_quality_motion_semantic_records。
# 如果 CLIP 依赖或模型权重不可用, records 会显式保留阻断原因, 不伪造语义通过。
cmd = probe_workflow.build_formal_metric_command(
    layout,
    semantic_model_id=SEMANTIC_MODEL_ID,
    semantic_frame_limit=SEMANTIC_FRAME_LIMIT,
    disable_semantic_metric=DISABLE_SEMANTIC_METRIC,
)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 10. 执行机制后处理
# 后处理只读取已有 records, 并写出 mechanism score、controlled negative、thresholds、reports 与 decision。
cmd = probe_workflow.build_mechanism_postprocess_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 11. 补齐 small-scale pilot 实验矩阵后处理
# 该步骤只从已有 generation / trajectory records 重建 governed pilot matrix records。
cmd = probe_workflow.build_pilot_matrix_postprocess_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 12. 执行真实 runtime 文件级攻击
# 该步骤读取 Drive run_root/videos 中的真实 mp4, 写出 attacked_videos 与 runtime_attack_records。
cmd = probe_workflow.build_runtime_attack_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 13. 执行 runtime attacked video 检测评分
# 该步骤把 attacked_videos 接入 detection records, 用于闭合工程链路。
cmd = probe_workflow.build_runtime_detection_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 14. 写出 small-scale claim pilot gate
# gate 只汇总 governed records, 不会把 proxy 或 heuristic threshold 伪装为最终论文 claim。
cmd = probe_workflow.build_small_scale_claim_pilot_gate_command(layout)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 15. 运行仓库 quick tests 与 harness 审计
# 注意: pytest 默认不运行重型 GPU 测试。这里检查治理约束和轻量功能测试。
!pytest -q
!python tools/harness/run_all_audits.py


In [ ]:
# 16. 打包并落盘到 Google Drive packages 子目录
# 打包逻辑由 scripts/package_results/generative_video_drive_packager.py 执行, Notebook 不直接写正式 package manifest。
cmd = probe_workflow.build_drive_packaging_command(layout, include_videos=INCLUDE_VIDEOS_IN_PACKAGE)
print(' '.join(cmd))
result = probe_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 17. 显示可下载审阅文件
package_dir = Path(layout['drive_package_dir'])
print('package_dir =', package_dir)
for path in sorted(package_dir.glob('*')):
    print(path)
